## Performance of NMWI


In [19]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import glob, warnings, sys, runpy
import matplotlib
matplotlib.use("cairo")
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, GridSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import matplotlib.ticker as mtick
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API")


In [2]:
# Run only once to get the working directory
project_root = Path("../../").resolve()
os.chdir(project_root)
print("Working dir:", Path.cwd())

Working dir: /Users/barrera.maria/Library/CloudStorage/GoogleDrive-barrera.maria1297@gmail.com/.shortcut-targets-by-id/1LkgWZVWWAMM27iFYdRSBNH8E7w0EPw9j/Maria B/Final_Codes


In [3]:
%matplotlib inline
# METADATA LOADING
vars_dict = runpy.run_path("Codes/Auxiliary_files/Generate_data_index.py")
globals().update(vars_dict)

Found 28 genus tables in index
Removed 4 OTU IDs listed in OTU_IDS_TO_EXCLUDE
Number of samples: 1807, and number of taxa: 294
Number of unique studies: 28

Sample_id per Category:
Category  n_sample_id
    case         1122
 control          685


- X: DataFrame (rows=samples, cols=taxonomy), index includes bioproject/sample_id RELATIVE ABUNDANCE
- y: Series/array-like of True/False (same index as X), True (1) =control, False=case

In [4]:
X_new = X.copy()

In [5]:
%run -i Codes/Auxiliary_files/NMWI.py # Run model

NMWI fitted; 10x10cv


In [6]:
%run -i Codes/Auxiliary_files/LOSO.py # Run model

In [7]:
###### RESULTS SUMMARY
print(f"10x10 CV BA: {scores.mean():.4f}") #print(f"10x10 CV SD: {scores.std():.4f}")
print(f"C: {C:.4f}")
print(f"LOSO: {mean_cv_ba:.4f}")
print(f"Train ba: {train_bal_acc:.4f}")
print(f"# positive coefs: {num_pos}, # negative coefs: {num_neg}, # zero coefs: {num_zero}")

10x10 CV BA: 0.7382
C: 0.7950
LOSO: 0.7272
Train ba: 0.7483
# positive coefs: 13, # negative coefs: 10, # zero coefs: 271


## Figure 3.a

In [20]:
## Leave-one-study-out 
plot_df = loso_df.sort_values("BA", ascending=False).reset_index(drop=True)
plot_df["Study_idx"] = np.arange(1, len(plot_df) + 1)

acc_pct = plot_df["BA"] * 100
mean_acc = acc_pct.mean()

# Dimensions
width_pt = 242
height_pt = 115
width_in = width_pt /26 ; height_in = height_pt / 26

fig, ax1 = plt.subplots(figsize=(width_in, height_in))
width = 0.6

# Stacked bars: positives + negatives per held-out study
ax1.bar(
    plot_df["Study_idx"], plot_df["pos_test"],
    width=width, color="#76c9ff", edgecolor="none", label="Healthy samples"
)
ax1.bar(
    plot_df["Study_idx"], plot_df["neg_test"],
    width=width, bottom=plot_df["pos_test"],
    color="#ff9fb5", edgecolor="none", label="Non-healthy samples"
)

ax1.set_xlabel("Study (in decreasing order of NMWI classification accuracy)")
ax1.set_ylabel("Number of samples per study (Bars)")
ax1.set_xlim(0.5, len(plot_df) + 0.5)
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.legend(loc="upper left", frameon=True)

# Line: accuracy per study
ax2 = ax1.twinx()
ax2.plot(
    plot_df["Study_idx"], acc_pct,
    "o-", linewidth=1.5, markersize=5, color="#f39c12"
)
ax2.set_ylabel("NMWI accuracy (%) per study (Points)")
ax2.set_ylim(0, 100)

# Mean accuracy reference line
ax2.axhline(mean_acc, linestyle="--", linewidth=1.2, color="#f39c12")
ax2.text(
    len(plot_df) - 3, mean_acc + 2,
    f"mean: {mean_acc:.1f}%",
    color="#f39c12", ha="left", va="bottom"
)

plt.tight_layout()

plt.savefig("Figures/Fig3a.pdf")
plt.show()

## NMWI FINAL MODEL

In [10]:
## lodo
temp_t = training_set.reset_index()
temp_t = temp_t.set_index(['BioProject.Number', 'Disease', 'Sample_id'])
X_lodo = temp_t[[c for c in temp_t.columns if c.startswith("d__")]]

disease_ids = np.array(X_lodo.index.get_level_values("Disease"))

lodo = LeaveOneGroupOut()
lodo_splits = list(lodo.split(X_lodo, y_arr, groups=disease_ids))
print("Total LODO folds (including healthy):", len(lodo_splits))

fold_bas = []
lodo_loop = []

for train_idx, test_idx in lodo_splits:
    disease_te = pd.Index(disease_ids[test_idx]).unique().tolist()[0]

    if str(disease_te) == "Healthy":
        continue

    X_tr = X_lodo.iloc[train_idx]
    y_tr = y_arr[train_idx]
    X_te = X_lodo.iloc[test_idx]
    y_te = y_arr[test_idx]

    disease_train = disease_ids[train_idx]
    disease_train_ser = pd.Series(disease_train)

    lodo_model = LogisticRegression(**final_params)
    lodo_model.fit(X_tr, y_tr)

    y_hat = lodo_model.predict(X_te)
    ba = balanced_accuracy_score(y_te, y_hat) 
    fold_bas.append(ba)

    pos_test = int(y_tr.sum())
    neg_test = int((~pd.Series(y_te.astype(bool))).sum())

    lodo_loop.append({
        "Disease": disease_te,
        "n_test": len(test_idx),
        "pos_test": pos_test,
        "neg_test": neg_test,
        "BA": ba 
    })

lodo_df = pd.DataFrame(lodo_loop).sort_values("BA")
print(lodo_df)

mean_cv_ba = float(np.mean(fold_bas)) if fold_bas else np.nan
print("LODO mean balanced accuracy (non-healthy diseases only):", mean_cv_ba)


Total LODO folds (including healthy): 8
    Disease  n_test  pos_test  neg_test        BA
5       GPA      68       685        68  0.352941
6        RA      21       685        21  0.380952
3       CRS     653       685       653  0.601838
1    Asthma     123       685       123  0.747967
0        AR     193       685       193  0.751295
4  CRS + AS      34       685        34  0.794118
2        CF      30       685        30  0.933333
LODO mean balanced accuracy (non-healthy diseases only): 0.6517778609370376


In [34]:
plot_df = lodo_df.sort_values("BA", ascending=False).reset_index(drop=True)
plot_df["Disease_idx"] = np.arange(1, len(plot_df) + 1)

acc_pct = plot_df["BA"] * 100
mean_acc = acc_pct.mean()

fig, ax1 = plt.subplots(figsize=(5, 4))

ax1.bar(
    plot_df["Disease_idx"],
    plot_df["neg_test"],              # non-healthy samples in test fold
    width=width,
    # bottom=plot_df["pos_test"],       # <<-- stack
    color="#ff9fb5",
    edgecolor="none",
    label="Non-healthy samples"
)

ax1.set_xlabel("Disease (in decreasing order of NMWI classification accuracy)")
ax1.set_ylabel("Number of samples per disease (Bars)")
ax1.set_xlim(0.5, len(plot_df) + 0.5)
ax1.set_ylim(0,700)
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

ax1.set_xticks(plot_df["Disease_idx"])
ax1.set_xticklabels(plot_df["Disease"], ha="right")

ax2 = ax1.twinx()
ax2.plot(
    plot_df["Disease_idx"],
    acc_pct,
    "o-",
    linewidth=1.5,
    markersize=5,
    color="#f39c12"
)

ax2.axhline(mean_acc, linestyle="--", linewidth=1.2, color="#f39c12")
ax2.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("Figures/Fig3c.pdf")
plt.show()

In [22]:
from sklearn.metrics import roc_curve, roc_auc_score

# Full model
full_auc = roc_auc_score(y.values, scores_all)
fpr_full, tpr_full, _ = roc_curve(y.values, scores_all)

# LOSO
loso_auc = roc_auc_score(y_arr, loso_scores)
fpr_loso, tpr_loso, _ = roc_curve(y_arr, loso_scores)

# 10x10cv
cv = RepeatedStratifiedKFold(
    n_splits=10,
    n_repeats=10,
    random_state=42
)

# each sample gets one test prediction per repeat, so we average them
cv_scores_sum = np.zeros(len(y))
cv_scores_count = np.zeros(len(y))

for train_idx, test_idx in cv.split(X_new, y.values):
    X_tr = X_new.iloc[train_idx]
    X_te = X_new.iloc[test_idx]
    y_tr = y.values[train_idx]

    model_10 = LogisticRegression(**final_params)
    model_10.fit(X_tr, y_tr)

    fold_scores = model_10.decision_function(X_te)
    cv_scores_sum[test_idx] += fold_scores
    cv_scores_count[test_idx] += 1

cv_scores_mean = cv_scores_sum / cv_scores_count

cv_auc = roc_auc_score(y.values, cv_scores_mean)
fpr_cv, tpr_cv, _ = roc_curve(y.values, cv_scores_mean)

# --------------------------------------------------
# 4. PLOT
# --------------------------------------------------
plt.figure(figsize=(6, 6))

plt.plot(fpr_full, tpr_full, lw=2, label=f"Full model (AUC = {full_auc:.3f})")
plt.plot(fpr_loso, tpr_loso, lw=2, label=f"LOSO (AUC = {loso_auc:.3f})")
plt.plot(fpr_cv, tpr_cv, lw=2, label=f"Repeated 10x10 CV (AUC = {cv_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig('Figures/Fig3b.pdf')
plt.show()

print(f'full AUC: {full_auc:.3f}')
print(f"LOSO (AUC = {loso_auc:.3f})")
print(f"Repeated 10x10 CV (AUC = {cv_auc:.3f})")

full AUC: 0.819
LOSO (AUC = 0.675)
Repeated 10x10 CV (AUC = 0.806)


### Figure 3.d

In [35]:
# Cutoff
interval = 0.1
max_cutoff = 1.5
cutoffs = np.arange(0, max_cutoff + interval, interval)
y_arr = y.values

In [36]:
# Because we reuse cv_splits and final_params, cutoff=0 now matches the same CV setup


results = {
    c: {"bal_acc": [], "recall": [], "tnr": [], "samples_retained": []}
    for c in cutoffs
}

# Build and store the exact repeated 10x10 splits once
n_splits, n_repeats = 10, 10


for tr, te in cv_splits:
    X_tr, X_te = X_new.iloc[tr], X_new.iloc[te]
    y_tr, y_te = y_arr[tr], y_arr[te]

    model_10 = LogisticRegression(**final_params)
    model_10.fit(X_tr, y_tr)

    fold_scores = model_10.decision_function(X_te)

    for c in cutoffs:
        # Keep only samples with score magnitude above cutoff
        keep = np.abs(fold_scores) >= c
        if not np.any(keep):
            continue

        # Use the filtered test set and model.predict() to avoid threshold mismatch
        X_keep = X_te.iloc[keep]
        y_keep = y_te[keep]
        y_pred = model_10.predict(X_keep)

        ba = balanced_accuracy_score(y_keep, y_pred)
        rec = recall_score(y_keep, y_pred, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_keep, y_pred, labels=[0, 1]).ravel()
        tnr = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        results[c]["bal_acc"].append(ba)
        results[c]["recall"].append(rec)
        results[c]["tnr"].append(tnr)
        results[c]["samples_retained"].append(keep.mean())
        

# Summarize across all folds
df_cutoff = pd.DataFrame([
    {
        "cutoff": c,
        "samples retained": np.nanmean(results[c]["samples_retained"]),
        "samples retained sd": np.nanstd(results[c]["samples_retained"]),
        "bal_acc": np.nanmean(results[c]["bal_acc"]),
        "bal_acc sd": np.nanstd(results[c]["bal_acc"]),
        "recall": np.nanmean(results[c]["recall"]),
        "tnr": np.nanmean(results[c]["tnr"]),
    }
    for c in cutoffs
])

display(df_cutoff)



,cutoff,samples retained,samples retained sd,bal_acc,bal_acc sd,recall,tnr
0,0.0,1.000000,0.000000,0.738208,0.027319,0.779437,0.696979
1,0.1,0.913997,0.019793,0.756982,0.026920,0.807315,0.706648
2,0.2,0.832695,0.029366,0.770920,0.028733,0.828463,0.713377
3,0.3,0.737892,0.035583,0.792552,0.031154,0.864051,0.721053
4,0.4,0.625940,0.042351,0.814565,0.032981,0.904167,0.724964
5,0.5,0.518311,0.041386,0.835577,0.034349,0.931504,0.739650
6,0.6,0.424906,0.041075,0.868500,0.033042,0.949811,0.787188
7,0.7,0.351306,0.035451,0.894801,0.033827,0.962602,0.827000
8,0.8,0.305094,0.034273,0.912529,0.034633,0.968232,0.856826
9,0.9,0.276153,0.032869,0.926367,0.032272,0.966469,0.886264


In [38]:
fig, ax1 = plt.subplots(figsize=(10, 5), dpi=200)
ax2 = ax1.twinx()

x = df_cutoff["cutoff"]

bal = df_cutoff["bal_acc"]
bal_sd = df_cutoff["bal_acc sd"]

ret = df_cutoff["samples retained"]
ret_sd = df_cutoff["samples retained sd"]

ax1.plot(x, bal, color="steelblue")
ax1.fill_between(
    x,
    bal - bal_sd,
    bal + bal_sd,
    color="steelblue",
    alpha=0.2
)

ax2.plot(x, ret, color="orange")
ax2.fill_between(
    x,
    ret - ret_sd,
    ret + ret_sd,
    color="orange",
    alpha=0.2
)

ax1.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

ax1.set_ylabel("10x10 CV balanced accuracy (%)", fontsize=15)
ax2.set_ylabel("Samples retained", fontsize=15)
ax1.set_xlabel("NMWI Magnitude Cutoff", fontsize=15)

plt.savefig('Figures/Fig3d.pdf')


In [47]:
def plot_cutoff_summary(df, title="LOSO", cutoffs=(0, 0.5, 1)):
    df = df.copy()
    df["cutoff"] = pd.to_numeric(df["cutoff"], errors="coerce")
    df_plot = df[df["cutoff"].isin(cutoffs)].sort_values("cutoff")

    print(df_plot)

    fig, ax1 = plt.subplots(figsize=(6, 6))

    bar_width = 0.35
    x = np.arange(len(df_plot))

    ax1.bar(
        x - bar_width / 2,
        df_plot["recall"],
        width=bar_width,
        label="Recall",
        color="#5DADE2",
        alpha=0.9
    )
    ax1.bar(
        x + bar_width / 2,
        df_plot["tnr"],
        width=bar_width,
        label="TNR",
        color="#F1948A",
        alpha=0.9
    )

    ax1.set_ylabel("Performance metrics", fontsize=12)
    ax1.set_ylim(0, 1)
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_plot["cutoff"], fontsize=12)
    ax1.set_xlabel("Cutoff", fontsize=12)

    ax2 = ax1.twinx()
    ax2.plot(
        x,
        df_plot["samples retained"] * 100,
        marker="o",
        color="darkorange",
        linewidth=2
    )
    ax2.set_ylabel("Proportion of retained samples (%)", color="darkorange", fontsize=12)
    ax2.set_ylim(0, 100)
    ax2.tick_params(axis="y", labelcolor="darkorange")

    ax1.legend(loc="upper left", fontsize=11)
    plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.savefig('Figures/Fig3e_2.pdf')
    plt.show()


In [44]:
plot_cutoff_summary(df_cutoff, title="10x10cv")

    cutoff  samples retained  samples retained sd   bal_acc  bal_acc sd  \
0      0.0          1.000000             0.000000  0.738208    0.027319   
5      0.5          0.518311             0.041386  0.835577    0.034349   
10     1.0          0.258387             0.031814  0.930146    0.033815   

      recall       tnr  
0   0.779437  0.696979  
5   0.931504  0.739650  
10  0.963116  0.897176  


In [46]:
## LOSO
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix

results = {
    c: {"bal_acc": [], "recall": [], "tnr": [], "samples_retained": []}
    for c in cutoffs
}

loso = LeaveOneGroupOut()
bio_ids = np.array(X_new.index.get_level_values(0))
y_arr = y.values.ravel()

loso_scores = np.full(len(y_arr), np.nan)
loso_loop = []
coef_rows = []
selected_sets = []
fold_bas = []
fold_sizes = []

for train_idx, test_idx in loso.split(X_new, y_arr, groups=bio_ids):
    X_tr = X_new.iloc[train_idx]
    X_te = X_new.iloc[test_idx]
    y_tr = y_arr[train_idx]
    y_te = y_arr[test_idx]

    model = LogisticRegression(**final_params)
    model.fit(X_tr, y_tr)

    fold_scores = model.decision_function(X_te)
    loso_scores[test_idx] = fold_scores

    y_hat = model.predict(X_te)
    ba = balanced_accuracy_score(y_te, y_hat)
    fold_bas.append(ba)
    fold_sizes.append(len(test_idx))

    proj_te = pd.Index(np.asarray(bio_ids)[test_idx]).unique().tolist()[0]

    loso_loop.append({
        "BioProject": proj_te,
        "n_test": len(test_idx),
        "pos_test": int(y_te.sum()),
        "neg_test": int((y_te == 0).sum()),
        "BA": ba
    })

    coefs = model.coef_.ravel()
    row = pd.Series(coefs, index=feature_names, name=proj_te)
    coef_rows.append(row)

    selected_sets.append(set(np.array(feature_names)[coefs != 0]))

    for c in cutoffs:
        keep = np.abs(fold_scores) >= c
        if not np.any(keep):
            continue

        y_keep = y_te[keep]
        y_pred = model.predict(X_te.iloc[keep])

        ba_c = balanced_accuracy_score(y_keep, y_pred)
        rec_c = recall_score(y_keep, y_pred, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_keep, y_pred, labels=[0, 1]).ravel()
        tnr_c = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        results[c]["bal_acc"].append(ba_c)
        results[c]["recall"].append(rec_c)
        results[c]["tnr"].append(tnr_c)
        results[c]["samples_retained"].append(keep.mean())

coef_df = pd.DataFrame(coef_rows)
loso_df = pd.DataFrame(loso_loop).sort_values("BA")
mean_cv_ba = float(np.mean(fold_bas)) if fold_bas else np.nan

df_cutoff_loso = pd.DataFrame([
    {
        "cutoff": c,
        "samples retained": np.nanmean(results[c]["samples_retained"]),
        "samples retained sd": np.nanstd(results[c]["samples_retained"]),
        "bal_acc": np.nanmean(results[c]["bal_acc"]),
        "bal_acc sd": np.nanstd(results[c]["bal_acc"]),
        "recall": np.nanmean(results[c]["recall"]),
        "tnr": np.nanmean(results[c]["tnr"]),
    }
    for c in cutoffs
])

display(df_cutoff_loso)

plot_cutoff_summary(df_cutoff_loso, title="LOSO")


,cutoff,samples retained,samples retained sd,bal_acc,bal_acc sd,recall,tnr
0,0.0,1.000000,0.000000,0.727191,0.172610,0.421231,0.707043
1,0.1,0.914381,0.086435,0.738147,0.172901,0.430982,0.710139
2,0.2,0.821856,0.131640,0.736267,0.190518,0.432889,0.696846
3,0.3,0.726153,0.182866,0.738075,0.194709,0.455328,0.672650
4,0.4,0.610810,0.237169,0.737779,0.191988,0.466805,0.662402
5,0.5,0.529845,0.266935,0.749373,0.189106,0.502938,0.654678
6,0.6,0.450540,0.292574,0.779822,0.204477,0.480040,0.690396
7,0.7,0.404134,0.299629,0.770210,0.246289,0.412668,0.750006
8,0.8,0.352127,0.301531,0.764929,0.268854,0.369907,0.758291
9,0.9,0.323738,0.281371,0.736797,0.288866,0.339826,0.745115


    cutoff  samples retained  samples retained sd   bal_acc  bal_acc sd  \
0      0.0          1.000000             0.000000  0.727191    0.172610   
5      0.5          0.529845             0.266935  0.749373    0.189106   
10     1.0          0.292874             0.275222  0.759116    0.284337   

      recall       tnr  
0   0.421231  0.707043  
5   0.502938  0.654678  
10  0.332860  0.754913  


In [48]:
### Full model

C = 0.795  # Best regularization parameter for relative abundance

final_params = dict(
    random_state=42,
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    C=C,
    max_iter=2000
)

NMWI = LogisticRegression(**final_params)
NMWI.fit(X_new, y.values.ravel())

y_arr = y.values.ravel()
y_train_pred = NMWI.predict(X_new)
scores_all = NMWI.decision_function(X_new)
train_bal_acc = balanced_accuracy_score(y_arr, y_train_pred)

coef = NMWI.coef_.flatten()
feature_names = X_new.columns

coefficients = pd.DataFrame(coef, index=feature_names, columns=["Coefficient"])
sorted_coefficients = coefficients.sort_values("Coefficient", ascending=False)
NMWI_coefs = sorted_coefficients[
    (sorted_coefficients["Coefficient"] > 0) | (sorted_coefficients["Coefficient"] < 0)
]
NMWI_coefs.reset_index().to_csv("NMWI_coefficients.csv", index=False)

num_pos = (sorted_coefficients["Coefficient"] > 0).sum()
num_neg = (sorted_coefficients["Coefficient"] < 0).sum()
num_zero = (sorted_coefficients["Coefficient"] == 0).sum()
num_coef = len(sorted_coefficients)

results_full = {
    c: {"bal_acc": [], "recall": [], "tnr": [], "samples_retained": []}
    for c in cutoffs
}

for c in cutoffs:
    keep = np.abs(scores_all) >= c
    if not np.any(keep):
        continue

    X_keep = X_new.iloc[keep]
    y_keep = y_arr[keep]
    y_pred = NMWI.predict(X_keep)

    ba = balanced_accuracy_score(y_keep, y_pred)
    rec = recall_score(y_keep, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_keep, y_pred, labels=[0, 1]).ravel()
    tnr = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    results_full[c]["bal_acc"].append(ba)
    results_full[c]["recall"].append(rec)
    results_full[c]["tnr"].append(tnr)
    results_full[c]["samples_retained"].append(keep.mean())

df_cutoff_full = pd.DataFrame([
    {
        "cutoff": c,
        "samples retained": np.nanmean(results_full[c]["samples_retained"]),
        "samples retained sd": np.nanstd(results_full[c]["samples_retained"]),
        "bal_acc": np.nanmean(results_full[c]["bal_acc"]),
        "bal_acc sd": np.nanstd(results_full[c]["bal_acc"]),
        "recall": np.nanmean(results_full[c]["recall"]),
        "tnr": np.nanmean(results_full[c]["tnr"]),
    }
    for c in cutoffs
])

display(df_cutoff_full)

plot_cutoff_summary(df_cutoff_full, title="full model")


,cutoff,samples retained,samples retained sd,bal_acc,bal_acc sd,recall,tnr
0,0.0,1.000000,0.0,0.748277,0.0,0.789781,0.706774
1,0.1,0.918096,0.0,0.768864,0.0,0.823718,0.714010
2,0.2,0.842280,0.0,0.778416,0.0,0.837790,0.719043
3,0.3,0.753182,0.0,0.803677,0.0,0.880081,0.727273
4,0.4,0.633647,0.0,0.827528,0.0,0.925659,0.729396
5,0.5,0.536248,0.0,0.846414,0.0,0.940000,0.752827
6,0.6,0.449917,0.0,0.879400,0.0,0.958042,0.800759
7,0.7,0.379081,0.0,0.913939,0.0,0.973214,0.854664
8,0.8,0.327061,0.0,0.926290,0.0,0.972678,0.879902
9,0.9,0.288877,0.0,0.933226,0.0,0.968944,0.897507


    cutoff  samples retained  samples retained sd   bal_acc  bal_acc sd  \
0      0.0          1.000000                  0.0  0.748277         0.0   
5      0.5          0.536248                  0.0  0.846414         0.0   
10     1.0          0.270061                  0.0  0.936072         0.0   

      recall       tnr  
0   0.789781  0.706774  
5   0.940000  0.752827  
10  0.965986  0.906158  
